
# GI Trials — ZIP Extraction, Coverage, and Enrollment (REVISED)

**What this notebook does** (Run All top-to-bottom):

1. **ZCTA setup**: loads 2020 TIGER/Line ZCTA geometries and builds a **valid US ZIP whitelist** + centroids.
2. **Trials combine & US site extraction**: loads your six CSVs, parses the **Locations** field, keeps **U.S. sites only**, validates ZIPs against ZCTAs, and deduplicates **(NCT, ZIP, cancer group)**.
3. **ZIP universe**: merges **RUCA 2020** (Urban/Rural) and **ACS 2023** (population, median household income, race/ethnicity). Also computes population-weighted income quartiles.
4. **Coverage**: computes **30/60 mile** coverage for **Any GI** and by cancer, including Urban/Rural, Income Q1/Q4, Hispanic, Black.
5. **Phase & Enrollment**: robust **phase normalization** (handles `PHASE1|PHASE2`, `EARLY_PHASE1`, roman numerals, A/B suffixes) and **enrollment totals** for Phase 1/2/3 per group and for **Any GI (unique NCT)**.
6. **QC & Validation**: checks for non-US leakage, invalid ZIPs, duplicates, geocoding completeness, monotonic coverage, and runs **sensitivity/uncertainty** tests.

**Key fixes vs prior versions**
- Tokenizes `Phases` like `PHASE1|PHASE2` and maps **Early Phase 1 → Phase 1**; **1/2 → 2**, **2/3 → 3**.
- Enforces **US-only site tokens** and validates **5-digit ZIPs** against ZCTA whitelist.
- Avoids pandas `FutureWarning` by using aggregation without `GroupBy.apply` on grouping columns.
- Writes clean outputs to the `out/` folder and provides light-weight QC artifacts for audit.

> ⚠️ Assumes your folder structure:
>
> - `data/` containing the six CSVs: `Colorectal.csv`, `Pancreatic Cancer.csv`, `Gastric Cancer.csv`, `Esophageal cancer.csv`, `Hepatocellular.csv`, `cholangiocarcinoma.csv`
> - `data/RUCA-codes-2020-zipcode.csv`
> - `tl_2020_us_zcta510/` (or `data/tl_2020_us_zcta510/`) with the ZCTA shapefile bundle.


In [8]:

# ==============================
# Config
# ==============================
import os, re, warnings, glob
import numpy as np
import pandas as pd

# DATA_DIR = "../data"
DATA_DIR = "split_cancer_data/"
# FILES = [
#     "Colorectal.csv",
#     "Pancreatic Cancer.csv",
#     "Gastric Cancer.csv",
#     "Esophageal cancer.csv",
#     "Hepatocellular.csv",
#     "cholangiocarcinoma.csv",
# ]

FILES = [
    "solid.csv",
    "leukemia.csv",
    "myeloma.csv",
    "lymphoma.csv"
]

# FILES = [
#     "Anal cancer_clean.csv",
#     "Appendiceal cancer.csv",
#     "Biliary Duct cancer_clean.csv",
#     "Colorectal cancer_clean.csv",
#     "Esophageal_Clean.csv",
#     "Gastric Cancer_clean.csv",
#     "Gastroesophageal_clean.csv",
#     "Hepatocellular cancer_clean.csv",
#     "Pancreatic_clean.csv",
#     "Small bowel adenocarcinoma_Clean.csv"
# ]

def _norm(s): 
    import re
    return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")

# F2KEY = {
#     _norm("Colorectal"):               "colorectal_cancer",
#     _norm("Pancreatic Cancer"):        "pancreatic_cancer",
#     _norm("Gastric Cancer"):           "gastric_cancer",
#     _norm("Esophageal cancer"):        "oesophageal_cancer",
#     _norm("Hepatocellular"):           "hepatocellular_cancer",
#     _norm("cholangiocarcinoma"):       "cholangiocarcinoma",
# }

# F2KEY = {
#     _norm("Anal cancer_clean"):               "anal_cancer",
#     _norm("Appendiceal cancer"):              "appendiceal_cancer",
#     _norm("Biliary Duct cancer_clean"):       "biliary_duct_cancer",
#     _norm("Colorectal cancer_clean"):         "colorectal_cancer",
#     _norm("Esophageal_Clean"):                "esophageal_cancer",
#     _norm("Gastric Cancer_clean"):            "gastric_cancer",
#     _norm("Gastroesophageal_clean"):          "gastroesophageal_cancer",
#     _norm("Hepatocellular cancer_clean"):     "hepatocellular_cancer",
#     _norm("Pancreatic_clean"):                "pancreatic_cancer",
#     _norm("Small bowel adenocarcinoma_Clean"): "adenocarcinoma",


# }
# LABELS = {
#     "anal_cancer":"Anal_cancer",
#     "appendiceal_cancer":"Appendiceal_cancer",
#     "biliary_duct_cancer":"BiliaryDuct_cancer",
#     "colorectal_cancer":"Colorectal_cancer",
#     "esophageal_cancer":"Esophageal_cancer",
#     "gastric_cancer":"Gastric_cancer",
#     "gastroesophageal_cancer":"Gastroesophageal_cancer",
#     "hepatocellular_cancer":"Hepatocellular_cancer",
#     "pancreatic_cancer":"Pancreatic_cancer",
#     "adenocarcinoma":"Adenocarcinoma"
    
    

# }


F2KEY = {
    _norm("solid"):               "Solid",
    _norm("leukemia"):            "Leukemia",
    _norm("myeloma"):             "Myeloma",
    _norm("lymphoma"):            "Lymphoma",
    
}

LABELS = {
    "Solid":"Solid_cancer",
    "Leukemia":"leukemia",
    "Myeloma":"myeloma",
    "Lymphoma":"lymphoma",
}


os.makedirs("out", exist_ok=True)


In [9]:

# ==============================
# PREREQ: ZCTA centroids + valid ZIP whitelist
# ==============================
import geopandas as gpd

CANDIDATE_DIRS = [
    "../data/tl_2020_us_zcta510"
]
ZCTA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(d)), None)
assert ZCTA_DIR is not None, f"ZCTA shapefile dir not found in: {CANDIDATE_DIRS}"

zcta = gpd.read_file(ZCTA_DIR).to_crs(epsg=4326)
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else (
         "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else None)
assert id_col is not None, f"Could not find ZCTA id column in {list(zcta.columns)[:10]}"

zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x
zcta["lat"] = zcta["centroid"].y

zip_centroids = zcta[[id_col, "lat", "lon"]].rename(columns={id_col:"zip"}).copy()
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)

valid_us_zips = set(zip_centroids["zip"].tolist())
print("zip_centroids:", zip_centroids.shape, "| valid_us_zips:", len(valid_us_zips))


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_4042/3573664915.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


zip_centroids: (33144, 3) | valid_us_zips: 33144


In [10]:

# ==============================
# Helpers: US site detection + ZIP extraction
# ==============================
import re

US_COUNTRY_PAT = re.compile(r"\b(united\s*states|u\.s\.a\.?|u\.s\.)\b", re.I)
US_STATE_ABBRS = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
    "ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"
}
US_STATE_NAMES = {
    "alabama","alaska","arizona","arkansas","california","colorado","connecticut","delaware",
    "district of columbia","florida","georgia","hawaii","idaho","illinois","indiana","iowa",
    "kansas","kentucky","louisiana","maine","maryland","massachusetts","michigan","minnesota",
    "mississippi","missouri","montana","nebraska","nevada","new hampshire","new jersey",
    "new mexico","new york","north carolina","north dakota","ohio","oklahoma","oregon",
    "pennsylvania","rhode island","south carolina","south dakota","tennessee","texas","utah",
    "vermont","virginia","washington","west virginia","wisconsin","wyoming"
}
ZIP5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")

def is_us_site(s: str) -> bool:
    if not isinstance(s, str): return False
    low = s.lower()
    if US_COUNTRY_PAT.search(low): return True
    m = re.findall(r",\s*([A-Z]{2})\b", s)
    if any(x in US_STATE_ABBRS for x in m): return True
    if any(name in low for name in US_STATE_NAMES): return True
    return False

def tokenize_locations_field(s: str):
    if not isinstance(s, str): return []
    return re.split(r"\s*\|\s*", s.strip())

def extract_zip(s: str):
    if not isinstance(s, str): return None
    m = ZIP5_RE.findall(s)
    return m[0] if m else None


In [11]:

# ==============================
# Combine CSVs + US-only site ZIPs + *_with_zip.csv
# ==============================
frames = []
sites_rows = []
out_dir = os.path.join(DATA_DIR, "output/car-t")
os.makedirs(out_dir, exist_ok=True)

for f in FILES:
    print(f)
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        warnings.warn(f"Missing file: {p}")
        continue
    df = pd.read_csv(p, dtype=str)


    # tolerant column mapping
    nct_col = next((c for c in df.columns if c.lower().startswith("nct")), None)
    phases_col = next((c for c in df.columns if c.lower()=="phases"), None)
    loc_col = "Locations" if "Locations" in df.columns else ("Location" if "Location" in df.columns else None)
    if nct_col is None or loc_col is None:
        raise ValueError(f"{f}: need NCT Number and Locations/Location columns")

    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")

    # master for later phase/enrollment work
    tmp_master = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "cancer_type": key,
        "phase_raw": df[phases_col].astype(str) if phases_col else np.nan
    })
    frames.append(tmp_master[["nct_id","cancer_type","phase_raw"]])

    # explode locations → tokens
    tok = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "locations": df[loc_col].astype(str)
    })
    tok["loc_list"] = tok["locations"].apply(tokenize_locations_field)
    tok = tok.explode("loc_list", ignore_index=True).dropna(subset=["loc_list"])

    # US-only + ZIP extraction + ZCTA whitelist
    tok = tok[tok["loc_list"].apply(is_us_site)]
    tok["zip"] = tok["loc_list"].apply(extract_zip).astype(str).str.zfill(5)
    tok = tok[tok["zip"].isin(valid_us_zips)]

    # save *_with_zip.csv preserving original columns and adding Zipcode
    out_path = os.path.join(out_dir, f"{os.path.splitext(os.path.basename(f))[0]}_with_zip.csv")
    df_out = df.copy()
    # create exploded rows aligned to tokens/zip
    exploded = tok[["nct_id","zip"]].dropna()
    exploded = exploded.drop_duplicates()
    # join back to original on NCT to duplicate rows per ZIP
    df_out = df_out.merge(exploded, left_on=nct_col, right_on="nct_id", how="inner").drop(columns=["nct_id"])
    df_out = df_out.rename(columns={"zip":"Zipcode"})
    df_out.to_csv(out_path, index=False)

    # rows for site_df
    tok["cancer_type"] = key
    tok = tok.drop_duplicates(subset=["nct_id","zip","cancer_type"])
    sites_rows.append(tok[["nct_id","cancer_type","zip"]])

# master + site_df + trial_sites
master = (pd.concat(frames, ignore_index=True)
          if frames else pd.DataFrame(columns=["nct_id","cancer_type","phase_raw"]))
master = master.sort_values(["nct_id","cancer_type"]).drop_duplicates(["nct_id","cancer_type"], keep="first")

site_df = (pd.concat(sites_rows, ignore_index=True)
           if sites_rows else pd.DataFrame(columns=["nct_id","cancer_type","zip"]))
site_df["zip"] = site_df["zip"].astype(str).str.zfill(5)

trial_sites = (site_df.drop_duplicates(["zip","cancer_type"])
               .merge(zip_centroids, on="zip", how="left")
               .dropna(subset=["lat","lon"]))

print("Master trials:", master.shape, "| Site rows (US ZIP validated):", site_df.shape, "| Unique geocoded sites:", len(trial_sites))


solid.csv
leukemia.csv
myeloma.csv
lymphoma.csv
Master trials: (189, 3) | Site rows (US ZIP validated): (704, 3) | Unique geocoded sites: 312


In [12]:

# ==============================
# Build ZIP/ZCTA universe (RUCA + ACS) -> zip_universe
# ==============================
import requests

ruca_path = os.path.join("../data", "RUCA-codes-2020-zipcode.csv")
ruca_raw = pd.read_csv(ruca_path, dtype=str)
def _normc(s): 
    import re; 
    return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")
ruca = ruca_raw.rename(columns={c: _normc(c) for c in ruca_raw.columns})

# pick RUCA and ZIP columns heuristically
cand_ruca_cols = [c for c in ruca.columns if ("primary" in c and "ruca" in c)]               or [c for c in ruca.columns if c.endswith("ruca") or c.startswith("ruca")]
assert cand_ruca_cols, f"Could not find a RUCA code column in {list(ruca.columns)[:10]}"
ruca_code_col = cand_ruca_cols[0]

zip_col = None
for c in ruca.columns:
    m = ruca[c].astype(str).str.extract(r"(\d{5})")[0]
    if m.notna().mean() >= 0.9:
        zip_col = c; break
assert zip_col, "Could not find a ZIP column in RUCA CSV."

def _ruca_to_urb(x: str) -> str:
    try:
        v = float(str(x).strip())
    except Exception:
        return "Unknown"
    if 1.0 <= v <= 3.0:  return "Urban"
    if 4.0 <= v <= 10.0: return "Rural"
    return "Unknown"

ruca_df = pd.DataFrame({
    "zip": ruca[zip_col].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5),
    "PrimaryRUCA": ruca[ruca_code_col].astype(str)
}).dropna(subset=["zip"]).drop_duplicates(subset=["zip"])
ruca_df["rural_urban"] = ruca_df["PrimaryRUCA"].apply(_ruca_to_urb)

# ACS fetch (2023); optional API key via env
YEAR = 2023
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "").strip()

def fetch_acs_vars(year: int, vars_list: list[str]) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": ",".join(["NAME"] + vars_list), "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()
    cols = rows[0]
    data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area":"zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    for v in vars_list:
        df[v] = pd.to_numeric(df[v], errors="coerce")
    return df[["zip"] + vars_list]

pop_df = fetch_acs_vars(YEAR, ["B01003_001E"]).rename(columns={"B01003_001E":"population"})
inc    = fetch_acs_vars(YEAR, ["B19013_001E"]).rename(columns={"B19013_001E":"mhi"})
eth    = fetch_acs_vars(YEAR, ["B03003_001E","B03003_003E"]).rename(
            columns={"B03003_001E":"pop_total_eth","B03003_003E":"pop_hispanic"})
blk    = fetch_acs_vars(YEAR, ["B02001_001E","B02001_003E"]).rename(
            columns={"B02001_001E":"pop_total_race","B02001_003E":"pop_black"})

zip_universe = (
    zip_centroids
      .merge(ruca_df[["zip","rural_urban"]], on="zip", how="left")
      .merge(pop_df, on="zip", how="left")
      .merge(inc,    on="zip", how="left")
      .merge(eth,    on="zip", how="left")
      .merge(blk,    on="zip", how="left")
)

for c in ["population","mhi","pop_total_eth","pop_hispanic","pop_total_race","pop_black"]:
    if c in zip_universe.columns:
        zip_universe[c] = pd.to_numeric(zip_universe[c], errors="coerce")

print("ZIP universe rows:", len(zip_universe),
      "| pop present:", f"{100*zip_universe['population'].notna().mean():.1f}%",
      "| mhi present:", f"{100*zip_universe['mhi'].notna().mean():.1f}%")

# Population-weighted income quartiles
def weighted_quantile(values, weights, qs):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w) - 0.5*w
    cw = cw / np.sum(w)
    return np.interp(qs, cw, v)

mask = zip_universe["mhi"].notna() & zip_universe["population"].notna()
if mask.any():
    q1,q2,q3 = weighted_quantile(zip_universe.loc[mask,"mhi"],
                                 zip_universe.loc[mask,"population"], [0.25,0.5,0.75])
    zip_universe["inc_quartile"] = pd.cut(zip_universe["mhi"],
                                          bins=[-np.inf,q1,q2,q3,np.inf],
                                          labels=["Q1 (Lowest)","Q2","Q3","Q4 (Highest)"])
else:
    zip_universe["inc_quartile"] = np.nan


ZIP universe rows: 33144 | pop present: 99.3% | mhi present: 99.3%


In [13]:

# ==============================
# Coverage @ 30 & 60 miles + stratified tables
# ==============================
from sklearn.neighbors import BallTree

for need in ["master","site_df","trial_sites","zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`")

zip_universe["population"] = pd.to_numeric(zip_universe["population"], errors="coerce")

EARTH_RADIUS_MI = 3958.7613
RADII_MI = [30, 60]

def build_tree(df):
    pts = np.deg2rad(df[["lat","lon"]].to_numpy())
    return BallTree(pts, metric="haversine")

trees = {}
if len(trial_sites):
    for ctype, g in trial_sites.groupby("cancer_type"):
        trees[ctype] = build_tree(g)
    tree_any = build_tree(trial_sites)
else:
    tree_any = None

qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

for r in RADII_MI:
    r_rad = r / EARTH_RADIUS_MI
    zip_universe[f"covered_any_gi_{r}mi"] = False if tree_any is None else         np.array([len(ix)>0 for ix in tree_any.query_radius(qpts, r=r_rad)], dtype=bool)
    for ctype in sorted(master["cancer_type"].unique()):
        t = trees.get(ctype)
        col = f"covered_{ctype}_{r}mi"
        zip_universe[col] = False if t is None else             np.array([len(ix)>0 for ix in t.query_radius(qpts, r=r_rad)], dtype=bool)

def pop_share(df, covered_col, weight_col="population"):
    if covered_col not in df or weight_col not in df or len(df)==0: return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def subpop_share(df, covered_col, subpop_col):
    if covered_col not in df or subpop_col not in df: return float("nan")
    w = pd.to_numeric(df[subpop_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def table_for_radius(r):
    rows = []
    keys = ["any_gi"] + list(master["cancer_type"].unique())
    for key in keys:
        label = LABELS.get(key, key)
        col = f"covered_any_gi_{r}mi" if key=="any_gi" else f"covered_{key}_{r}mi"
        pop_all  = pop_share(zip_universe, col)
        pop_urb  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Urban"], col)
        pop_rur  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Rural"], col)
        q1       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q1 (Lowest)"], col)
        q4       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q4 (Highest)"], col)
        hisp     = subpop_share(zip_universe, col, "pop_hispanic")
        black    = subpop_share(zip_universe, col, "pop_black")
        rows.append({
            "Cancer Type": label,
            "% Pop Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
            "% Urban Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
            "% Rural Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
            "% Q1 Income Covered": 100*q1 if pd.notna(q1) else np.nan,
            "% Q4 Income Covered": 100*q4 if pd.notna(q4) else np.nan,
            "% Hispanic Covered": 100*hisp if pd.notna(hisp) else np.nan,
            "% Black Covered": 100*black if pd.notna(black) else np.nan,
        })
    return pd.DataFrame(rows)

table_30 = table_for_radius(30)
table_60 = table_for_radius(60)

table_30.to_csv(os.path.join("out", "coverage_30mi.csv"), index=False)
table_60.to_csv(os.path.join("out", "coverage_60mi.csv"), index=False)
table_60.head(10)


,Cancer Type,% Pop Covered,% Urban Covered,% Rural Covered,% Q1 Income Covered,% Q4 Income Covered,% Hispanic Covered,% Black Covered
0,any_gi,72.854692,80.584027,34.379267,54.177945,93.219386,74.149888,79.923961
1,lymphoma,69.949432,77.889857,30.423147,50.131647,92.001752,71.825346,77.862107
2,Solid_cancer,51.126788,58.136162,16.234238,33.138432,76.981464,53.768540,57.737439
3,myeloma,67.566328,75.576898,27.690779,46.799534,91.132463,70.579709,73.946969
4,leukemia,43.041848,49.388348,11.452591,25.146714,68.610906,51.955460,46.164993


In [14]:

# ==============================
# Phase 1/2/3: Trials & Enrollment (per group + Any GI single row)
# ==============================
# Robust parser: tokenizes 'PHASE1|PHASE2', handles EARLY_PHASE1, roman numerals, A/B suffix
import re

ROMAN = {"I":1,"II":2,"III":3,"IV":4}

def _tokenize_phases(s: str):
    if not isinstance(s, str) or not s.strip():
        return []
    t = s.upper().strip()
    t = re.sub(r'[\s,;/&]+', '|', t)
    t = t.replace('||','|').strip('|')
    return [tok for tok in t.split('|') if tok]

def _phase_int_from_token(tok: str):
    if "EARLY" in tok and "PHASE1" in tok:
        return 1
    if "PHASE0" in tok:
        return 1
    m = re.search(r'PHASE[_\s-]*([0-4]|I{1,3}|IV)\s*[AB]?$', tok)
    if not m:
        m = re.match(r'^([0-4]|I{1,3}|IV)$', tok)
    if not m:
        return None
    g = m.group(1)
    v = int(g) if g.isdigit() else ROMAN.get(g, None)
    return v if v in {1,2,3,4} else None

def normalize_phase(phases_text: str):
    toks = _tokenize_phases(phases_text)
    vals = [v for v in (_phase_int_from_token(t) for t in toks) if v is not None]
    if not vals:
        return None
    st = set(vals)
    if st == {1,2}: v = 2
    elif st == {2,3}: v = 3
    else: v = max(vals)
    return f"Phase {v}"

# Build enroll_df from source CSVs (pulls Enrollment if present)
def parse_enrollment(x):
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"(\d[\d,]*)", s)
    if not m: return np.nan
    return pd.to_numeric(m.group(1).replace(",",""), errors="coerce")

rows = []
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p, dtype=str)
    nct_col    = next((c for c in df.columns if c.lower().startswith("nct")), None)
    phases_col = next((c for c in df.columns if c.lower()=="phases"), None)
    enroll_col = next((c for c in df.columns if c.lower()=="enrollment"), None)
    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")
    tmp = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "cancer_type": key,
        "phase_norm": df[phases_col].apply(normalize_phase) if phases_col else np.nan,
        "enrollment_num": df[enroll_col].apply(parse_enrollment) if enroll_col else np.nan,
    })
    rows.append(tmp)

enroll_df = (pd.concat(rows, ignore_index=True)
             if rows else pd.DataFrame(columns=["nct_id","cancer_type","phase_norm","enrollment_num"]))
enroll_df = enroll_df.sort_values(["nct_id","cancer_type"]).drop_duplicates(["nct_id","cancer_type"], keep="first")

phase_order = ["Phase 1","Phase 2","Phase 3"]

counts = (enroll_df.dropna(subset=["phase_norm"])
          .groupby(["cancer_type","phase_norm"])["nct_id"].nunique()
          .unstack(fill_value=0)
          .reindex(columns=phase_order, fill_value=0))
counts.columns = [f"Trials {ph}" for ph in counts.columns]

enr = (enroll_df.groupby(["cancer_type","phase_norm"])["enrollment_num"]
       .sum(min_count=1)
       .unstack()
       .reindex(columns=phase_order))
enr.columns = [f"Enrollment {ph}" for ph in enr.columns]

per_group = pd.concat([counts, enr], axis=1).reset_index()
per_group.insert(0, "Cancer Type", per_group["cancer_type"].map(LABELS).fillna(per_group["cancer_type"]))
per_group = per_group.drop(columns=["cancer_type"])

def phase_to_int(ph):
    return {"Phase 1":1, "Phase 2":2, "Phase 3":3, "Phase 4":4}.get(ph, np.nan)

gi_unique = (enroll_df
             .assign(phase_int=lambda d: d["phase_norm"].map(phase_to_int))
             .groupby("nct_id", as_index=False)
             .agg(phase_int=("phase_int","max"), enrollment_num=("enrollment_num","max")))
gi_unique["phase_norm"] = gi_unique["phase_int"].map({1:"Phase 1",2:"Phase 2",3:"Phase 3",4:"Phase 4"})

any_counts = gi_unique.groupby("phase_norm")["nct_id"].nunique()
any_enr    = gi_unique.groupby("phase_norm")["enrollment_num"].sum(min_count=1)

row = {"Cancer Type": "Any GI (Unique NCT)"}
for ph in phase_order:
    row[f"Trials {ph}"]     = int(any_counts.get(ph, 0))
    val = any_enr.get(ph, np.nan)
    row[f"Enrollment {ph}"] = float(val) if pd.notna(val) else np.nan
any_gi_one_row = pd.DataFrame([row])

per_group.to_csv(os.path.join("out","phase_enrollment_by_group.csv"), index=False)
any_gi_one_row.to_csv(os.path.join("out","phase_enrollment_any_gi_unique_row.csv"), index=False)
per_group.head(10), any_gi_one_row


(    Cancer Type  Trials Phase 1  Trials Phase 2  Trials Phase 3  \
 0      leukemia              24              10               0   
 1      lymphoma              43              23               2   
 2       myeloma              16               5               2   
 3  Solid_cancer              55               9               0   
 
    Enrollment Phase 1  Enrollment Phase 2  Enrollment Phase 3  
 0               737.0               609.0                 NaN  
 1              1780.0              2456.0               508.0  
 2              1019.0               501.0               890.0  
 3              2180.0              1346.0                 NaN  ,
            Cancer Type  Trials Phase 1  Enrollment Phase 1  Trials Phase 2  \
 0  Any GI (Unique NCT)             138              5716.0              47   
 
    Enrollment Phase 2  Trials Phase 3  Enrollment Phase 3  
 0              4912.0               4              1398.0  )

In [15]:

# ==============================
# QC battery (slim): invalid ZIPs, duplicates, geocoding, monotonic coverage
# ==============================
print("[QC] site_df rows:", len(site_df), "| unique (NCT,ZIP,group):", len(site_df.drop_duplicates(['nct_id','zip','cancer_type'])))
bad_len = (site_df["zip"].str.len()!=5).sum()
not_in_whitelist = (~site_df["zip"].isin(valid_us_zips)).sum()
dups = site_df.duplicated(subset=["nct_id","zip","cancer_type"]).sum()
print("  zip length !=5:", bad_len, "| zip not in whitelist:", not_in_whitelist, "| duplicates:", dups)

# Geocoding completeness
miss_geo = trial_sites["lat"].isna().sum() + trial_sites["lon"].isna().sum()
print("[QC] trial_sites geocoding missing (sum):", miss_geo)

# Monotonicity 30->60
cov30 = [c for c in zip_universe.columns if c.endswith("_30mi")]
cov60 = [c for c in zip_universe.columns if c.endswith("_60mi")]
if cov30 and cov60:
    probs = []
    for c30 in cov30:
        c60 = c30.replace("_30mi","_60mi")
        if c60 in zip_universe:
            bad = (zip_universe[c30] & ~zip_universe[c60]).sum()
            if bad: probs.append((c30, int(bad)))
    if probs:
        print("[QC] PROBLEM: 30mi covered but not at 60mi:", probs)
    else:
        print("[QC] OK: 60mi is a superset of 30mi for all flags")
else:
    print("[QC] Skipped monotonicity (flags not present)")


[QC] site_df rows: 704 | unique (NCT,ZIP,group): 704
  zip length !=5: 0 | zip not in whitelist: 0 | duplicates: 0
[QC] trial_sites geocoding missing (sum): 0
[QC] OK: 60mi is a superset of 30mi for all flags


In [16]:
# Replace the subgroup share helper with the correct logic:
def subgroup_coverage(df, covered_col: str, subgroup_col: str) -> float:
    """
    Share of a subgroup (e.g., Hispanic or Black) that lives in covered ZIPs.
    Numerator and denominator both use the subgroup column.
    """
    if any(col not in df.columns for col in [covered_col, subgroup_col]) or df.empty:
        return float("nan")
    w = pd.to_numeric(df[subgroup_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)


In [32]:
# ==============================
# Page-2 style coverage table (like the screenshot)
# Prereqs: master, trial_sites (with lat/lon), zip_universe (with lat/lon, population, mhi, pop_hispanic, pop_black, rural_urban, inc_quartile)
# ==============================
import numpy as np
import pandas as pd

# ---- Params
RADIUS_MI = 30   # <- set 30 or 60
INCLUDE_OTHER_GI = False

# ---- Label map (adjust if you added/removed groups)
CANCER_LABELS = LABELS
if INCLUDE_OTHER_GI:
    CANCER_LABELS["other_gi"] = "Other GI"

# ---- Safety checks
for need in ["master","trial_sites","zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`. Run the combine/ZIP-universe steps first.")

# Ensure numerics present
zip_universe["population"]   = pd.to_numeric(zip_universe["population"], errors="coerce")
zip_universe["pop_hispanic"] = pd.to_numeric(zip_universe.get("pop_hispanic", np.nan), errors="coerce")
zip_universe["pop_total_eth"]= pd.to_numeric(zip_universe.get("pop_total_eth", np.nan), errors="coerce")
zip_universe["pop_black"]    = pd.to_numeric(zip_universe.get("pop_black", np.nan), errors="coerce")
zip_universe["pop_total_race"]=pd.to_numeric(zip_universe.get("pop_total_race", np.nan), errors="coerce")

# ---- If covered_* columns for this radius are missing, compute them
def ensure_coverage_flags(radius_mi: int):
    col_any = f"covered_any_gi_{radius_mi}mi"
    cols_needed = [col_any] + [f"covered_{k}_{radius_mi}mi" for k in master["cancer_type"].unique()]
    missing = [c for c in cols_needed if c not in zip_universe.columns]
    if not missing:
        return
    from sklearn.neighbors import BallTree
    EARTH_RADIUS_MI = 3958.7613
    r_rad = radius_mi / EARTH_RADIUS_MI
    qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

    def tree_from(df):
        if df.empty: 
            return None
        pts = np.deg2rad(df[["lat","lon"]].to_numpy())
        return BallTree(pts, metric="haversine")

    # Any GI
    t_any = tree_from(trial_sites)
    zip_universe[col_any] = False if t_any is None else np.array([len(ix)>0 for ix in t_any.query_radius(qpts, r=r_rad)], bool)

    # Per cancer
    for key, g in trial_sites.groupby("cancer_type"):
        t = tree_from(g)
        col = f"covered_{key}_{radius_mi}mi"
        zip_universe[col] = False if t is None else np.array([len(ix)>0 for ix in t.query_radius(qpts, r=r_rad)], bool)

ensure_coverage_flags(RADIUS_MI)

# ---- Helpers
def pop_share(df: pd.DataFrame, covered_col: str, weight_col: str = "population") -> float:
    if covered_col not in df.columns or weight_col not in df.columns or df.empty:
        return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def covered_population_millions(df: pd.DataFrame, covered_col: str) -> float:
    if covered_col not in df.columns or "population" not in df.columns:
        return float("nan")
    w = pd.to_numeric(df["population"], errors="coerce").fillna(0.0)
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / 1_000_000.0)

def race_eth_share(df: pd.DataFrame, covered_col: str, num_col: str, den_col: str) -> float:
    for col in (covered_col, num_col, den_col):
        if col not in df.columns:
            return float("nan")
    num = pd.to_numeric(df[num_col], errors="coerce").fillna(0.0)
    den = pd.to_numeric(df[den_col], errors="coerce").fillna(0.0).sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((num[c]).sum() / den)

def counts_for(key: str) -> tuple[int, int]:
    if key == "any_gi":
        n_trials = int(master["nct_id"].nunique())
        n_sites  = int(trial_sites["zip"].nunique())
        return n_trials, n_sites
    tf = master[master["cancer_type"] == key]
    ts = trial_sites[trial_sites["cancer_type"] == key]
    return int(tf["nct_id"].nunique()), int(ts["zip"].nunique())

# ---- Build rows
rows = []
ordered_keys = ["any_gi","bispecific","car"]
if INCLUDE_OTHER_GI: ordered_keys.append("other_gi")
# keep only those keys that actually exist in your data (besides any_gi)
exists = set(master["cancer_type"].unique())
ordered_keys = ["any_gi"] + [k for k in ordered_keys[1:] if k in exists] + [k for k in exists if k not in ordered_keys]

for key in ordered_keys:
    label = CANCER_LABELS.get(key, key)
    covered_col = f"covered_any_gi_{RADIUS_MI}mi" if key=="any_gi" else f"covered_{key}_{RADIUS_MI}mi"
    n_trials, n_sites = counts_for(key)

    # overall + subgroups
    pop_all  = pop_share(zip_universe, covered_col, "population")
    coveredM = covered_population_millions(zip_universe, covered_col)

    if "rural_urban" in zip_universe.columns:
        pop_urb = pop_share(zip_universe[zip_universe["rural_urban"]=="Urban"], covered_col)
        pop_rur = pop_share(zip_universe[zip_universe["rural_urban"]=="Rural"], covered_col)
    else:
        pop_urb = pop_rur = float("nan")

    if "inc_quartile" in zip_universe.columns:
        q1_df = zip_universe[zip_universe["inc_quartile"]=="Q1 (Lowest)"]
        q4_df = zip_universe[zip_universe["inc_quartile"]=="Q4 (Highest)"]
        pop_q1 = pop_share(q1_df, covered_col)
        pop_q4 = pop_share(q4_df, covered_col)
    else:
        pop_q1 = pop_q4 = float("nan")

# Old (incorrect):
# hisp = race_eth_share(zip_universe, covered_col, "pop_hispanic", "pop_total_eth")
# blk  = race_eth_share(zip_universe, covered_col, "pop_black", "pop_total_race")

# New (correct):
    hisp = subgroup_coverage(zip_universe, covered_col, "pop_hispanic")
    blk  = subgroup_coverage(zip_universe, covered_col, "pop_black")


    rows.append({
        "Cancer Type": label,
        "No. Trials (NCT IDs)": n_trials if n_trials>0 else np.nan,
        "No. Sites (ZIPs)": n_sites if n_sites>0 else np.nan,
        "% Population Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
        "Covered Population (M)": coveredM if pd.notna(coveredM) else np.nan,
        "% Patients Covered (SEER Incidence)": np.nan,  # placeholder
        "% Urban Pop Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
        "% Rural Pop Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
        "% Lowest Income Quartile Covered": 100*pop_q1 if pd.notna(pop_q1) else np.nan,
        "% Highest Income Quartile Covered": 100*pop_q4 if pd.notna(pop_q4) else np.nan,
        "% Hispanic Pop Covered": 100*hisp if pd.notna(hisp) else np.nan,
        "% Black Pop Covered": 100*blk if pd.notna(blk) else np.nan,
    })

table = pd.DataFrame(rows)

# ---- Nicely formatted version (matches the screenshot style)
fmt = table.copy()
pct_cols = [
    "% Population Covered","% Patients Covered (SEER Incidence)","% Urban Pop Covered",
    "% Rural Pop Covered","% Lowest Income Quartile Covered","% Highest Income Quartile Covered",
    "% Hispanic Pop Covered","% Black Pop Covered",
]
for c in pct_cols:
    if c in fmt.columns:
        fmt[c] = fmt[c].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "NA")

if "Covered Population (M)" in fmt.columns:
    fmt["Covered Population (M)"] = fmt["Covered Population (M)"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")

# Save and display
out_path = f"out/page2_table_{RADIUS_MI}mi.csv"
fmt.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
fmt


Saved: out/page2_table_30mi.csv


,Cancer Type,No. Trials (NCT IDs),No. Sites (ZIPs),% Population Covered,Covered Population (M),% Patients Covered (SEER Incidence),% Urban Pop Covered,% Rural Pop Covered,% Lowest Income Quartile Covered,% Highest Income Quartile Covered,% Hispanic Pop Covered,% Black Pop Covered
0,any_gi,189,169,56.7%,189.57,NA,66.9%,5.9%,36.8%,82.3%,62.6%,68.0%
1,leukemia,34,39,30.5%,102.08,NA,36.4%,1.4%,18.1%,53.0%,39.9%,36.1%
2,myeloma,23,96,50.6%,169.37,NA,60.1%,3.4%,31.6%,77.7%,56.8%,62.0%
3,Solid_cancer,64,54,37.1%,123.98,NA,44.2%,1.5%,22.7%,61.3%,41.2%,46.6%
4,lymphoma,68,123,53.5%,178.86,NA,63.2%,4.8%,34.0%,79.9%,59.9%,65.1%


In [19]:
# --- TOP 10 ZIP CODES BY TRIAL CONCENTRATION ---

# 1. Count unique trials per ZIP
zip_concentration = (site_df.groupby("zip")["nct_id"].nunique()
                     .sort_values(ascending=False)
                     .reset_index(name="Unique Trial Count"))

# 2. Merge with Centroids to identify the city/location if possible 
# (You can manually cross-reference these ZIPs or use a geocoding library)
top_10_zips = zip_concentration.head(10)

print("--- Top 10 ZIP Codes by Trial Concentration ---")
display(top_10_zips)

# 3. Save for your report
top_10_zips.to_csv("out/top_10_trial_zip_codes.csv", index=False)

--- Top 10 ZIP Codes by Trial Concentration ---


,zip,Unique Trial Count
0,77030,44
1,91010,27
2,19104,20
3,10065,19
4,33612,17
5,63110,14
6,30322,13
7,02215,13
8,37203,12
9,98109,12


In [35]:
import pandas as pd
import os

# 1. Calculate Concentration per Cancer Type
concentration_results = []

# Get the list of cancer types from your master dataframe
cancer_types = master['cancer_type'].unique()

for ctype in cancer_types:
    # Filter sites for this specific cancer
    ctype_sites = site_df[site_df['cancer_type'] == ctype].copy()
    total_sites = len(ctype_sites)
    
    if total_sites == 0:
        continue
        
    # Count occurrences of each ZIP
    top_zips = ctype_sites['zip'].value_counts().head(10).reset_index()
    top_zips.columns = ['zip', 'trial_count']
    
    # Calculate Percentage of total trials for this cancer type
    top_zips['pct_of_total_sites'] = (top_zips['trial_count'] / total_sites) * 100
    top_zips['cancer_type'] = LABELS.get(ctype, ctype)
    
    # 2. Merge with zip_universe for Demographic Analysis
    # This adds Rural/Urban, Income Quartile, and Population data
    enriched_zips = top_zips.merge(
        zip_universe[['zip', 'rural_urban', 'inc_quartile', 'population', 'mhi']], 
        on='zip', 
        how='left'
    )
    
    concentration_results.append(enriched_zips)

# Combine into one master analysis table
concentration_df = pd.concat(concentration_results, ignore_index=True)

# 3. Save the results
os.makedirs("out/analysis", exist_ok=True)
concentration_df.to_csv("out/analysis/top_10_zips_concentration.csv", index=False)

# Display Example (Top 10 for the first cancer type)
print("Top ZIP Code Concentrations (Sample):")
display(concentration_df.head(10))

Top ZIP Code Concentrations (Sample):


,zip,trial_count,pct_of_total_sites,cancer_type,rural_urban,inc_quartile,population,mhi
0,77030,14,5.090909,lymphoma,Urban,Q3,12976.0,79415.0
1,91010,9,3.272727,lymphoma,Urban,Q3,27245.0,98189.0
2,53226,7,2.545455,lymphoma,Urban,Q3,18893.0,86361.0
3,66205,6,2.181818,lymphoma,Urban,Q3,13728.0,100139.0
4,84112,5,1.818182,lymphoma,Urban,Q1 (Lowest),2738.0,-666666666.0
5,52242,5,1.818182,lymphoma,Urban,Q1 (Lowest),4412.0,-666666666.0
6,78229,5,1.818182,lymphoma,Urban,Q1 (Lowest),35195.0,46718.0
7,80218,5,1.818182,lymphoma,Urban,Q3,19364.0,82925.0
8,43210,5,1.818182,lymphoma,Urban,Q1 (Lowest),11853.0,30197.0
9,19104,5,1.818182,lymphoma,Urban,Q1 (Lowest),54454.0,39526.0


In [36]:
import pandas as pd

# 1. Total unique ZIP codes across ALL trials combined
total_unique_zips = site_df['zip'].nunique()

# 2. Unique ZIP codes per cancer type
unique_zips_per_cancer = site_df.groupby('cancer_type')['zip'].nunique().reset_index()
unique_zips_per_cancer.columns = ['Cancer Type', 'Unique ZIP Count']

# Map the labels for readability
unique_zips_per_cancer['Cancer Type'] = unique_zips_per_cancer['Cancer Type'].map(LABELS)

# 3. Print the results
print("--- Geographic Coverage Summary ---")
print(f"Total unique ZIP codes across all files: {total_unique_zips}")
print("\nBreakdown by Cancer Type:")
print(unique_zips_per_cancer.to_string(index=False))

# Optional: Export this summary to your output folder
unique_zips_per_cancer.to_csv("out/unique_zip_summary.csv", index=False)

--- Geographic Coverage Summary ---
Total unique ZIP codes across all files: 169

Breakdown by Cancer Type:
 Cancer Type  Unique ZIP Count
    leukemia                39
    lymphoma               123
     myeloma                96
Solid_cancer                54


In [40]:
import pandas as pd
import os

# 1. Pool all site data from the 4 categories
# site_df already contains 'nct_id', 'zip', and 'cancer_type' from your 4 files
all_cart_sites = site_df.copy()

# 2. Get the unique set of trials (Deduplicated across all files)
total_unique_nct_ids = all_cart_sites['nct_id'].nunique()

# 3. Calculate True Top 10 ZIP Concentration
# We count how many unique trials are present in each ZIP, regardless of cancer type
zip_concentration = all_cart_sites.groupby('zip')['nct_id'].nunique().sort_values(ascending=False).head(10).reset_index()
zip_concentration.columns = ['zip', 'unique_trial_count']

# Percentage based on the total deduplicated trial pool
zip_concentration['pct_of_total_cart_trials'] = (zip_concentration['unique_trial_count'] / total_unique_nct_ids) * 100

# Merge with demographics from your zip_universe
top_10_zips_combined = zip_concentration.merge(
    zip_universe[['zip', 'rural_urban', 'inc_quartile', 'mhi']], 
    on='zip', 
    how='left'
)

# 4. Calculate True Top 10 Institutional Density
# We use the 'car_t_tokens' from the previous step which contains the institution names
# Deduplicating so an institution with 1 trial appearing in 2 files is only counted once
top_10_institutions_combined = car_t_tokens.groupby('institution').agg(
    unique_trials=('nct_id', 'nunique'),
    unique_zips=('zip', 'nunique')
).sort_values(by='unique_trials', ascending=False).head(10).reset_index()

top_10_institutions_combined['density_pct'] = (top_10_institutions_combined['unique_trials'] / total_unique_nct_ids) * 100

# --- Verification Printout ---
print(f"VERIFIED COMBINED ANALYSIS")
print(f"Total Unique CAR-T Trials (Deduplicated): {total_unique_nct_ids}")
print("-" * 30)
print("\nTop 10 ZIP Codes (Combined CAR-T Landscape):")
display(top_10_zips_combined)

print("\nTop 10 Institutions (Combined CAR-T Landscape):")
display(top_10_institutions_combined)

VERIFIED COMBINED ANALYSIS
Total Unique CAR-T Trials (Deduplicated): 165
------------------------------

Top 10 ZIP Codes (Combined CAR-T Landscape):


,zip,unique_trial_count,pct_of_total_cart_trials,rural_urban,inc_quartile,mhi
0,77030,44,26.666667,Urban,Q3,79415.0
1,91010,27,16.363636,Urban,Q3,98189.0
2,19104,20,12.121212,Urban,Q1 (Lowest),39526.0
3,10065,19,11.515152,Urban,Q4 (Highest),160938.0
4,33612,17,10.303030,Urban,Q1 (Lowest),43919.0
5,63110,14,8.484848,Urban,Q2,73809.0
6,30322,13,7.878788,Urban,Q1 (Lowest),-666666666.0
7,02215,13,7.878788,Urban,Q2,65827.0
8,37203,12,7.272727,Urban,Q2,69671.0
9,98109,12,7.272727,Urban,Q4 (Highest),130845.0



Top 10 Institutions (Combined CAR-T Landscape):


,institution,unique_trials,unique_zips,density_pct
0,Houston Methodist Hospital,7,1,4.242424
1,Banner MD Anderson Cancer Center,5,1,3.030303
2,Cedars-Sinai Medical Center,4,1,2.424242
3,Colorado Blood Cancer Institute,4,1,2.424242
4,City of Hope Medical Center,4,1,2.424242
5,City of Hope,4,1,2.424242
6,Stanford University,4,2,2.424242
7,Texas Transplant Institute,4,1,2.424242
8,Memorial Sloan Kettering Cancer Center,3,1,1.818182
9,MD Anderson Cancer Center,3,1,1.818182


In [43]:
import pandas as pd
import numpy as np

# Step 1: Combine all site data from the 4 cancer categories
combined_cart_landscape = site_df.copy()

# Step 1.1: Normalize ZIP types (important for merges + leading zeros)
combined_cart_landscape["zip"] = combined_cart_landscape["zip"].astype(str).str.extract(r"(\d{5})")[0]
combined_cart_landscape = combined_cart_landscape.dropna(subset=["zip"])
combined_cart_landscape["zip"] = combined_cart_landscape["zip"].astype(str).str.zfill(5)

zip_universe["zip"] = zip_universe["zip"].astype(str).str.extract(r"(\d{5})")[0]
zip_universe = zip_universe.dropna(subset=["zip"])
zip_universe["zip"] = zip_universe["zip"].astype(str).str.zfill(5)

# Step 1.2: Clean ACS sentinel / invalid income values (fixes -666666666.0, etc.)
ACS_SENTINELS = {-666666666, -999999999, -888888888, -333333333}

zip_universe["mhi"] = pd.to_numeric(zip_universe["mhi"], errors="coerce")
zip_universe.loc[zip_universe["mhi"].isin(ACS_SENTINELS), "mhi"] = np.nan
zip_universe.loc[zip_universe["mhi"] <= 0, "mhi"] = np.nan   # optional guard

# Step 2: True unique trial count across entire landscape (dedupe across categories)
total_deduplicated_trials = combined_cart_landscape["nct_id"].nunique()

# Step 3: Concentration by ZIP (unique trials per ZIP)
top_10_zip_concentration = (
    combined_cart_landscape.groupby("zip")["nct_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="n_trials")
)

# Step 4: % of total CAR-T landscape
top_10_zip_concentration["pct_of_total_cart_landscape"] = (
    top_10_zip_concentration["n_trials"] / total_deduplicated_trials * 100
)

# Step 5: Merge demographics
final_analysis = top_10_zip_concentration.merge(
    zip_universe[["zip", "rural_urban", "inc_quartile", "mhi"]],
    on="zip",
    how="left"
)

# (Optional) nicer formatting for display
final_analysis["pct_of_total_cart_landscape"] = final_analysis["pct_of_total_cart_landscape"].round(2)
final_analysis["mhi"] = final_analysis["mhi"].round(0)

print(f"Total Unique Trials in Combined Pool: {total_deduplicated_trials}")
display(final_analysis)


Total Unique Trials in Combined Pool: 165


,zip,n_trials,pct_of_total_cart_landscape,rural_urban,inc_quartile,mhi
0,77030,44,26.67,Urban,Q3,79415.0
1,91010,27,16.36,Urban,Q3,98189.0
2,19104,20,12.12,Urban,Q1 (Lowest),39526.0
3,10065,19,11.52,Urban,Q4 (Highest),160938.0
4,33612,17,10.30,Urban,Q1 (Lowest),43919.0
5,63110,14,8.48,Urban,Q2,73809.0
6,30322,13,7.88,Urban,Q1 (Lowest),NaN
7,02215,13,7.88,Urban,Q2,65827.0
8,37203,12,7.27,Urban,Q2,69671.0
9,98109,12,7.27,Urban,Q4 (Highest),130845.0


In [49]:
import os
import pandas as pd
import numpy as np

# --- Institution extractor: takes first chunk before first comma ---
def extract_institution(loc_str: str):
    if not isinstance(loc_str, str):
        return None
    s = loc_str.strip()
    if not s:
        return None
    return s.split(",")[0].strip() or None

site_inst_rows = []

for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        print(f"[WARN] Missing file: {p}")
        continue

    df = pd.read_csv(p, dtype=str)

    nct_col = next((c for c in df.columns if c.lower().startswith("nct")), None)
    loc_col = "Locations" if "Locations" in df.columns else ("Location" if "Location" in df.columns else None)
    if nct_col is None or loc_col is None:
        raise ValueError(f"{f}: need NCT and Locations/Location columns")

    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")

    tok = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "locations": df[loc_col].astype(str)
    })

    tok["loc_list"] = tok["locations"].apply(tokenize_locations_field)
    tok = tok.explode("loc_list", ignore_index=True).dropna(subset=["loc_list"])

    # US-only + ZIP extraction + whitelist
    tok = tok[tok["loc_list"].apply(is_us_site)].copy()
    tok["zip"] = tok["loc_list"].apply(extract_zip)
    tok = tok.dropna(subset=["zip"])
    tok["zip"] = tok["zip"].astype(str).str.zfill(5)
    tok = tok[tok["zip"].isin(valid_us_zips)].copy()

    # Institution column
    tok["institution"] = tok["loc_list"].apply(extract_institution)

    tok["cancer_type"] = key
    tok = tok.dropna(subset=["institution"])
    tok = tok.drop_duplicates(subset=["nct_id","institution","zip","cancer_type"])

    site_inst_rows.append(tok[["nct_id","institution","zip","cancer_type"]])

site_inst_df = pd.concat(site_inst_rows, ignore_index=True)
print("site_inst_df:", site_inst_df.shape, "| unique institutions:", site_inst_df["institution"].nunique())


site_inst_df: (724, 4) | unique institutions: 356


In [66]:
import numpy as np
import pandas as pd

# -------------------------------
# Top institutions by unique trials
# -------------------------------
top10_inst = (
    site_inst_df.drop_duplicates(["nct_id","institution"])
    .groupby("institution")["nct_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(358)
    .reset_index(name="n_trials")
)

total_unique_trials = site_inst_df["nct_id"].nunique()
top10_inst["pct_of_unique_trials"] = 100 * top10_inst["n_trials"] / total_unique_trials

# -------------------------------
# HHI on institutional site shares (trial×institution×zip occurrences)
# -------------------------------
inst_site_counts = (
    site_inst_df
    .drop_duplicates(["nct_id","institution","zip"])  # correct site-level unit
    .groupby("institution")
    .size()
)

inst_share = inst_site_counts / inst_site_counts.sum()
HHI_inst = float(np.sum((inst_share * 100) ** 2))

# -------------------------------
# Add HHI summary to table
# -------------------------------
# top10_inst["HHI_institution_site_shares"] = HHI_inst
# top10_inst["HHI_interpretation"] = (
#     "HIGHLY CONCENTRATED" if HHI_inst > 2500 
#     else "MODERATELY CONCENTRATED" if HHI_inst >= 1500 
#     else "NOT highly concentrated"
# )

# -------------------------------
# Display results
# -------------------------------
display(top10_inst)



# -------------------------------
# Save table
# -------------------------------
top10_inst.to_csv("out/top_institutions_with_HHI.csv", index=False)
print("Saved to: out/top_institutions_with_HHI.csv")


,institution,n_trials,pct_of_unique_trials
0,City of Hope Medical Center,15,9.090909
1,Texas Children's Hospital,14,8.484848
2,Memorial Sloan Kettering Cancer Center,11,6.666667
3,Moffitt Cancer Center,11,6.666667
4,Houston Methodist Hospital,10,6.060606
...,...,...,...
351,Mayo Clinic in Arizona,1,0.606061
352,Mayo Clinic Hospital,1,0.606061
353,Mayo Clinic Florida,1,0.606061
354,Mayo Clinic Arizona,1,0.606061


Saved to: out/top_institutions_with_HHI.csv


In [51]:
import numpy as np

# Use institution-site footprint (NOT just trial count)
inst_site_counts = (
    site_inst_df
    .drop_duplicates(["nct_id", "institution", "zip"])  # each delivery site
    .groupby("institution")
    .size()
    .sort_values(ascending=False)
)

total_sites = inst_site_counts.sum()
inst_share = inst_site_counts / total_sites

# HHI on SITE SHARES (correct market structure measure)
HHI_inst = float(np.sum((inst_share * 100) ** 2))

print("\n=== HHI (Institutional DELIVERY site shares) ===")
print(f"Total CAR-T delivery site occurrences: {total_sites}")
print(f"HHI (0–10,000 scale): {HHI_inst:.1f}")

if HHI_inst > 2500:
    print("Interpretation: HIGHLY CONCENTRATED (policy-relevant)")
elif HHI_inst > 1500:
    print("Interpretation: MODERATELY CONCENTRATED")
else:
    print("Interpretation: DISPERSED (but still elite-center clustered)")



=== HHI (Institutional DELIVERY site shares) ===
Total CAR-T delivery site occurrences: 724
HHI (0–10,000 scale): 125.1
Interpretation: DISPERSED (but still elite-center clustered)


In [57]:
# ============================================================
# Population coverage within 30 miles of EACH institution
# (Using your existing ZIP centroids + population in zip_universe)
#
# Requires in your notebook:
# - site_inst_df (columns: nct_id, institution, zip, ...)
# - zip_universe (columns: zip, lat, lon, population, ...)
#   (you already created zip_universe from zip_centroids + ACS)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# ----------------------------
# 0) Safety checks
# ----------------------------
for need in ["site_inst_df", "zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`. Make sure you've built it earlier.")

# ----------------------------
# 1) Clean/standardize ZIP + population
# ----------------------------
def norm_zip(s):
    s = pd.Series(s).astype(str).str.extract(r"(\d{5})")[0]
    return s.astype(str).str.zfill(5)

site_inst_df = site_inst_df.copy()
zip_universe = zip_universe.copy()

site_inst_df["zip"] = norm_zip(site_inst_df["zip"])
zip_universe["zip"] = norm_zip(zip_universe["zip"])

site_inst_df = site_inst_df.dropna(subset=["zip", "institution"])
zip_universe = zip_universe.dropna(subset=["zip", "lat", "lon"])

zip_universe["population"] = pd.to_numeric(zip_universe["population"], errors="coerce").fillna(0.0)

TOTAL_POP = float(zip_universe["population"].sum())
if TOTAL_POP <= 0:
    raise RuntimeError("Total population is 0/missing in zip_universe. Check ACS pull / population column.")

# ----------------------------
# 2) Attach lat/lon to each institution site using ZIP centroids
# ----------------------------
site_geo = site_inst_df.merge(
    zip_universe[["zip", "lat", "lon"]],
    on="zip",
    how="left"
).dropna(subset=["lat", "lon"])

# Optional: dedupe to unique institution-site points (institution, zip)
site_geo = site_geo.drop_duplicates(["institution", "zip"])

# ----------------------------
# 3) Compute population covered within 30 miles of each institution
# ----------------------------
EARTH_RADIUS_MI = 3958.7613
RADIUS_MI = 30
r_rad = RADIUS_MI / EARTH_RADIUS_MI

# Query points = all ZIP centroids
qpts = np.deg2rad(zip_universe[["lat", "lon"]].to_numpy())
pop = zip_universe["population"].to_numpy()

def build_tree(df_points):
    pts = np.deg2rad(df_points[["lat", "lon"]].to_numpy())
    return BallTree(pts, metric="haversine")

rows = []
for inst, g in site_geo.groupby("institution"):
    # A single institution can have multiple ZIP sites; use all as coverage anchors
    t = build_tree(g)
    covered_idx = t.query_radius(qpts, r=r_rad)
    covered_mask = np.array([len(ix) > 0 for ix in covered_idx], dtype=bool)

    covered_pop = float(pop[covered_mask].sum())
    covered_pop_share = 100.0 * covered_pop / TOTAL_POP

    rows.append({
        "institution": inst,
        "n_site_zips_for_inst": int(g["zip"].nunique()),
        "covered_population": covered_pop,
        "covered_population_M": covered_pop / 1_000_000.0,
        "covered_population_share_%": covered_pop_share
    })

inst_coverage = pd.DataFrame(rows).sort_values("covered_population_share_%", ascending=False)

print(f"Computed 30-mile coverage for {len(inst_coverage):,} institutions.")
display(inst_coverage.head(20))

# ----------------------------
# 4) (Optional) Focus on your TOP-10 institutions by trial density
# If you already have top10_inst from earlier, this uses it.
# Otherwise it computes top10 by unique trials quickly.
# ----------------------------
if "top10_inst" not in globals():
    top10_inst = (
        site_inst_df.drop_duplicates(["nct_id", "institution"])
        .groupby("institution")["nct_id"].nunique()
        .sort_values(ascending=False)
        .head(10)
        .reset_index(name="n_trials")
    )

top10_names = set(top10_inst["institution"].tolist())
top10_cov = inst_coverage[inst_coverage["institution"].isin(top10_names)] \
    .merge(top10_inst, on="institution", how="left") \
    .sort_values("n_trials", ascending=False)

print("\n=== 30-mile population coverage for Top-10 institutions (by trials) ===")
display(top10_cov)

# ----------------------------
# 5) Save outputs
# ----------------------------
import os
os.makedirs("out", exist_ok=True)
inst_coverage.to_csv("out/institution_30mi_population_coverage_all.csv", index=False)
top10_cov.to_csv("out/institution_30mi_population_coverage_top10.csv", index=False)
print("\nSaved:")
print(" - out/institution_30mi_population_coverage_all.csv")
print(" - out/institution_30mi_population_coverage_top10.csv")


Computed 30-mile coverage for 356 institutions.


,institution,n_site_zips_for_inst,covered_population,covered_population_M,covered_population_share_%
206,Research Site,41,113975421.0,113.975421,34.064180
287,University of California,3,17142221.0,17.142221,5.123348
145,Memorial Sloan Kettering Cancer Center,3,15896769.0,15.896769,4.751116
348,Weill Cornell Medical College - NY Presbyteria...,1,15758304.0,15.758304,4.709732
148,Memorial Sloan Kettering Cancer Center (MSKCC)...,1,15758304.0,15.758304,4.709732
349,Weill Cornell Medicine - New York Presbyterian...,1,15758304.0,15.758304,4.709732
118,Local Institution - 0229,1,15727376.0,15.727376,4.700489
146,Memorial Sloan Kettering Cancer Center (All Pr...,1,15727376.0,15.727376,4.700489
147,Memorial Sloan Kettering Cancer Center (All pr...,1,15727376.0,15.727376,4.700489
152,Memorial Sloan Kettering Comprehensive Cancer ...,1,15727376.0,15.727376,4.700489



=== 30-mile population coverage for Top-10 institutions (by trials) ===


,institution,n_site_zips_for_inst,covered_population,covered_population_M,covered_population_share_%,n_trials,pct_of_unique_trials
1,City of Hope Medical Center,1,10820250.0,10.820250,3.233881,15,9.090909
5,Texas Children's Hospital,1,6094843.0,6.094843,1.821584,14,8.484848
0,Memorial Sloan Kettering Cancer Center,3,15896769.0,15.896769,4.751116,11,6.666667
8,Moffitt Cancer Center,2,3297357.0,3.297357,0.985491,11,6.666667
3,Houston Methodist Hospital,1,6094843.0,6.094843,1.821584,10,6.060606
4,MD Anderson Cancer Center,1,6094843.0,6.094843,1.821584,9,5.454545
6,Stanford University,2,4961599.0,4.961599,1.482888,9,5.454545
7,Banner MD Anderson Cancer Center,1,3394667.0,3.394667,1.014574,9,5.454545
9,St. Jude Children's Research Hospital,1,1232446.0,1.232446,0.368345,9,5.454545
2,Children's Hospital of Philadelphia,2,6471681.0,6.471681,1.934211,8,4.848485



Saved:
 - out/institution_30mi_population_coverage_all.csv
 - out/institution_30mi_population_coverage_top10.csv


In [58]:
import pandas as pd

# -----------------------------------
# Count unique trials per institution
# -----------------------------------
inst_trial_counts = (
    site_inst_df.drop_duplicates(["nct_id", "institution"])
    .groupby("institution")["nct_id"]
    .nunique()
)

# -----------------------------------
# Institutions with only 1 unique trial
# -----------------------------------
n_inst_total = inst_trial_counts.shape[0]
n_inst_single_trial = (inst_trial_counts == 1).sum()

pct_single_trial = (n_inst_single_trial / n_inst_total) * 100

print(f"Total institutions: {n_inst_total}")
print(f"Institutions with ONLY 1 unique trial: {n_inst_single_trial}")
print(f"Percentage of institutions with ONLY 1 trial: {pct_single_trial:.2f}%")


Total institutions: 356
Institutions with ONLY 1 unique trial: 246
Percentage of institutions with ONLY 1 trial: 69.10%


In [59]:
trial_count_distribution = (
    inst_trial_counts
    .value_counts()
    .sort_index()
    .reset_index()
)

trial_count_distribution.columns = ["n_trials", "n_institutions"]
trial_count_distribution["pct_of_institutions"] = (
    trial_count_distribution["n_institutions"] /
    trial_count_distribution["n_institutions"].sum()
) * 100

display(trial_count_distribution)


,n_trials,n_institutions,pct_of_institutions
0,1,246,69.101124
1,2,47,13.202247
2,3,25,7.022472
3,4,13,3.651685
4,5,9,2.528090
5,6,2,0.561798
6,7,3,0.842697
7,8,2,0.561798
8,9,4,1.123596
9,10,1,0.280899


In [60]:
import pandas as pd

# -------------------------------------------------
# Step 1: Build enrollment lookup from raw trial files
# -------------------------------------------------
enrollment_rows = []

for f in FILES:
    p = os.path.join(DATA_DIR, f)
    df = pd.read_csv(p, dtype=str)

    nct_col = next((c for c in df.columns if c.lower().startswith("nct")), None)
    enroll_col = next((c for c in df.columns if "enroll" in c.lower()), None)

    if nct_col is None or enroll_col is None:
        print(f"[SKIP] {f} missing NCT or Enrollment")
        continue

    tmp = df[[nct_col, enroll_col]].copy()
    tmp.columns = ["nct_id", "enrollment"]

    tmp["enrollment"] = pd.to_numeric(tmp["enrollment"], errors="coerce").fillna(0)

    enrollment_rows.append(tmp)

enrollment_df = pd.concat(enrollment_rows, ignore_index=True)

# Deduplicate NCT enrollment
enrollment_df = enrollment_df.drop_duplicates("nct_id")

print("Enrollment records:", enrollment_df.shape)

# -------------------------------------------------
# Step 2: Merge enrollment onto institution-site table
# -------------------------------------------------
inst_enroll = site_inst_df.merge(
    enrollment_df,
    on="nct_id",
    how="left"
)

inst_enroll["enrollment"] = inst_enroll["enrollment"].fillna(0)

# -------------------------------------------------
# Step 3: Aggregate total enrollment per institution
# -------------------------------------------------
inst_enroll_totals = (
    inst_enroll.groupby("institution")["enrollment"]
    .sum()
    .sort_values(ascending=False)
)

total_enrollment = inst_enroll_totals.sum()

top10_inst_enroll = (
    inst_enroll_totals.head(10)
    .reset_index(name="total_enrollment")
)

top10_inst_enroll["pct_of_total_enrollment"] = (
    top10_inst_enroll["total_enrollment"] / total_enrollment * 100
)

print("\n=== Top 10 Institutions by TOTAL Enrollment ===")
display(top10_inst_enroll)

print(f"\nTotal enrollment across all trials: {int(total_enrollment):,}")


Enrollment records: (189, 2)

=== Top 10 Institutions by TOTAL Enrollment ===


,institution,total_enrollment,pct_of_total_enrollment
0,Research Site,4735,4.020037
1,Moffitt Cancer Center,2415,2.050346
2,Mayo Clinic,2384,2.024027
3,Banner MD Anderson Cancer Center,2370,2.012141
4,Colorado Blood Cancer Institute,1903,1.615656
5,University of California,1622,1.377085
6,City of Hope,1585,1.345672
7,Huntsman Cancer Institute,1470,1.248037
8,Fred Hutchinson Cancer Center,1424,1.208982
9,Swedish Cancer Institute,1347,1.143609



Total enrollment across all trials: 117,785


In [67]:
site_inst_df = site_inst_df[
    ~site_inst_df["institution"].str.lower().str.contains("research site|unknown|site only|not available")
]


In [68]:
inst_enroll_totals = (
    inst_enroll.groupby("institution")["enrollment"]
    .sum()
    .sort_values(ascending=False)
)

top10_inst_enroll = inst_enroll_totals.head(10).reset_index(name="total_enrollment")
top10_inst_enroll["pct_of_total_enrollment"] = (
    top10_inst_enroll["total_enrollment"] / inst_enroll_totals.sum() * 100
)

display(top10_inst_enroll)


,institution,total_enrollment,pct_of_total_enrollment
0,Research Site,4735,4.020037
1,Moffitt Cancer Center,2415,2.050346
2,Mayo Clinic,2384,2.024027
3,Banner MD Anderson Cancer Center,2370,2.012141
4,Colorado Blood Cancer Institute,1903,1.615656
5,University of California,1622,1.377085
6,City of Hope,1585,1.345672
7,Huntsman Cancer Institute,1470,1.248037
8,Fred Hutchinson Cancer Center,1424,1.208982
9,Swedish Cancer Institute,1347,1.143609
